# 3. Working With Data & Visualisation

By the end of this session you should be able to:
- Load data into a Pandas DataFrame from a CSV file and inspect its contents
- Use basic built-in Pandas operations
- Create plots using Matplotlib (line plots, scatter plots, histograms)

## Loading data with Pandas

When working with data, you will mostly encounter data stored in plain text formats. The two most common are:

- CSV (Comma-Separated Values): tabular data where each row is a line and each column is separated by a comma
- TXT: plain text files, which may use other delimiters such as spaces or tabs

The most used Python library for loading and working with tabular data is Pandas (see https://pandas.pydata.org/docs/index.html for the documentation). Pandas represents data as a DataFrame, which is a two-dimensional table with labelled rows and columns, similar to a spreadsheet. To load a CSV file into a DataFrame you use the `.read_csv()` function, where you are required to specify the path to the file containing the data.

In [ ]:
import pandas as pd

The dataset we will use is NASA's Global Surface Temperature Analysis (GISTEMP), which records monthly global mean temperature anomalies relative to the 1951–1980 average, from 1880 to the present. It is publicly available from NASA GISS. Before we begin, download the dataset from NASA GISS:

1. Go to https://data.giss.nasa.gov/gistemp/data_v4.html
2. Under **Tables of Global and Hemispheric Monthly Means and Zonal Annual Means**, click CSV next to Global-mean monthly, seasonal, and annual means
3. Save the file (right-click on the link and select "Download Linked File") as GLB.Ts+dSST.csv in the same folder as this notebook

In [ ]:
df = pd.read_csv('GLB.Ts+dSST.csv', skiprows=1) # skiprows skips the first row of the file
                                                # which contains a header comment rather than column names

**Note:** `.txt` files can also be loaded with `pd.read_csv()` by specifying the delimiter via the sep argument (e.g. `sep="\s+"` for whitespace-separated files). However, they may require additional arguments depending on the file's structure.

## Analysing Data with Pandas

To visualise a DataFrame, we can use `.head()`, where the first 5 rows of data will be shown by default.

In [ ]:
df.head()

You can also visualise the last 5 rows using `.tail()`.

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.describe()

Some other commands:
- `.shape`: number of rows and columns
- `.columns`: see the column names
- `.dtypes`: see the data type of each column

Individual rows can be accessed by their index using `.loc[]`, although we will not use row access extensively in this session. To access specific columns of a DataFrame:

In [ ]:
df["Jan"]

In [ ]:
df[["Feb", "Mar"]] # note that you need double square brackets

In [ ]:
df["January"]

Sometimes you will encounter issues with the DataFrame entries such as missing data. Before performing operations on a DataFrame, you might need to clean the data.

In [ ]:
# replace missing placeholders
df.replace("***", pd.NA, inplace=True)

# drop rows where annual mean is missing
df_clean = df.dropna(subset=["J-D"])
df_clean["J-D"] = pd.to_numeric(df_clean["J-D"])

You can also filter according to conditions:

In [ ]:
# years that had a positive annual temperature anomaly
df_clean[df_clean["J-D"] > 0]

In [ ]:
# data from 1980 onwards
df_clean[df_clean["Year"] >= 1980]

DataFrame columns behave similarly to NumPy arrays. You can perform arithmetic operations directly on them:

In [ ]:
# The dataset stores anomalies in units of 0.01°C — convert to °C
df_clean["J-D"] / 100

# Add a new column with the anomaly in °C
df_clean["Annual_C"] = df_clean["J-D"] / 100

In [ ]:
# Average of two months
df_clean["Jan_Feb_mean"] = (df_clean["Jan"] + df_clean["Feb"]) / 2

Pandas provides built-in methods for common summary statistics, which can be applied directly to a column:

In [ ]:
print("Mean anomaly:", df_clean["Annual_C"].mean())
print("Max anomaly: ", df_clean["Annual_C"].max())
print("Min anomaly: ", df_clean["Annual_C"].min())

`groupby` allows you to split the data into groups and apply an aggregation function to each group. For example, to compute the average annual temperature anomaly per decade:

In [ ]:
# Create a decade column
df_clean["Decade"] = (df_clean["Year"] // 10) * 10

# Average anomaly per decade
df_clean.groupby("Decade")["Annual_C"].mean()

## Data Visualisation with Matplotlib

Matplotlib is the standard Python library for data visualisation. We will use `pyplot`, which is the main submodule used for plotting graphs.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

We can use `matplotlib.pyplot` for graphs like line plots (`plt.line`), scatter plots (`plt.scatter`) or histograms (`plt.hist`).

In [ ]:
plt.plot(df_clean["Year"], df_clean["Annual_C"])
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.title("Global Mean Temperature Anomaly")
plt.show()

In [ ]:
plt.scatter(df_clean["Year"], df_clean["Annual_C"])
plt.xlabel("Year")
plt.ylabel("Temperature Anomaly (°C)")
plt.title("Global Mean Temperature Anomaly")
plt.show()

In [ ]:
plt.hist(df_clean["Annual_C"], bins=30)
plt.xlabel("Temperature Anomaly (°C)")
plt.ylabel("Frequency")
plt.title("Distribution of Annual Temperature Anomalies")
plt.show()

### Customising plots

By default, Matplotlib automatically assigns a different colour to each line plotted on the same graph. You can override this, as well as customise the line style and width. For a full list of named colours, linestyles, and other options, see the Matplotlib documentation https://matplotlib.org/stable/gallery/index.html.

To identify each line, assign it a label using the `label` argument and call `plt.legend()` after your plot calls. This will automatically display a legend using the labels you have defined.

In [ ]:
# x axis values
x = np.linspace(0, 2 * np.pi, 100)

# line plots
plt.plot(x, np.sin(x), color="blue", linewidth=2, label=r"$\sin(x)$") # you can use the r before a string to insert latex (without it you get an error)
plt.plot(x, np.cos(x), color="red", linewidth=2, linestyle="--", label=r"$\cos(x)$")
plt.plot(x, np.sin(2 * x), color="seagreen", linewidth=1.5, linestyle=":", label=r"$\sin(2x)$")

plt.xlabel("x")
plt.ylabel("y")
plt.title("Trigonometric Functions")
plt.legend()
plt.grid(True)
plt.show()

To display several plots side by side, use `plt.subplots()`. This creates a figure containing several panels called axes. You specify the number of rows and columns of panels, and Matplotlib returns a fig object (the overall figure) and an axes array (one entry per panel).

To plot into a specific panel, use `axes[i]` instead of `plt.`. Note that the method names change slightly.

In [ ]:
x = np.linspace(0, 2 * np.pi, 100)

# one row, two columns:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# left plot
axes[0].plot(x, np.sin(x), color="steelblue", linewidth=2)
axes[0].set_title(r"$\sin(x)$")
axes[0].set_xlabel(r"$x$")
axes[0].set_ylabel(r"$y$")
axes[0].grid(True)

# right plot
axes[1].plot(x, np.exp(-x) * np.sin(5 * x), color="red", linewidth=2)
axes[1].set_title(r"Damped oscillation: $e^{-x}\sin(5x)$")
axes[1].set_xlabel(r"$x$")
axes[1].set_ylabel(r"$y$")
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Exercises


For the following exercises, assume the dataset has already been loaded and cleaned as in the teaching notebook. Use the DataFrame `df_clean` with the `Annual_C` column in degrees Celsius.

1. Compute the month with the highest mean temperature anomaly across all years. Compute the month which has the lowest.

In [ ]:
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# why do we need this step first?
for month in months:
    df_clean[month] = pd.to_numeric(df_clean[month], errors="coerce")


2. Write a function `monthly_means(df)` that returns a dictionary mapping each month name to its mean anomaly across all years.

3. Using the monthly means computed in Q2, produce a bar plot (see https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.bar.html) showing the mean anomaly for each month. Add appropriate axis labels and a title.

4. Pick any three years of your choice and plot their monthly anomaly profiles (Jan–Dec) as three lines on one graph. Add a legend and grid.

5. Write a function `fourier_square(x, N)` that takes an array of $x$ values and a number of terms $N$ and returns the Fourier series approximation:

$$ f(x) = \frac{4}{\pi} \sum_{k \in \{1,3,5,\ldots \}}^{N} \frac{\sin(kx)}{k} $$

6. Create a figure with four subplots $(2 \times 2)$ showing the approximation for $N = 1, 5, 25, 100$. Use the same $y$-axis limits across all subplots.